Note: This notebook calls modular functions from `src` folder in sequential order.

In [8]:
# Import sys
import sys
sys.path.append("../src")

# Import modules
from data_utils import load_and_clean_data, split_features_target
from preprocessing import create_preprocessing_pipeline
from models import create_models, train_and_evaluate
from mlflow_utils import get_data_version, create_comprehensive_metrics, log_model, compare_and_log_models

print("✅ Modules imported successfully")

✅ Modules imported successfully


# 1. Data loading and cleaning
Cleaning consists of dropping duplicates and replacing NaNs with the mean.

In [9]:
print("🔷 Loading and cleaning data...")
df_silver = load_and_clean_data("../data/raw/Data_set_cas_1.csv")

🔷 Loading and cleaning data...
Raw data loaded shape: (78, 62)
Cleaned data shape (duplicates dropped/NaNs replaced with mean): (72, 62)


# 2. Train/Test/Split

In [10]:
print("🔷 Split data into train and test sets...")
X_train, X_test, y_train, y_test = split_features_target(df_silver)

🔷 Split data into train and test sets...
Train shape: (57, 61), Test shape: (15, 61)


# 3. Preprocessing pipeline

In [11]:
print("🔷 Create preprocessing pipeline (scikit-learn)...")
preproc = create_preprocessing_pipeline()

🔷 Create preprocessing pipeline (scikit-learn)...
Preprocessing pipeline created


# 4. Model creation

In [12]:
print("🔷 Creating model pipelines (Elastic and Lasso)...")
pipeline_lasso, pipeline_enc = create_models(preproc)

🔷 Creating model pipelines (Elastic and Lasso)...
Model pipelines created


# 5. Training and evaluation

In [13]:
# Train Lasso model (best performing one)
lasso_model, lasso_pred, lasso_r2 = train_and_evaluate(
    pipeline_lasso, X_train, y_train, X_test, y_test, "Lasso")

# Train Elastic model (
enc_model, enc_pred, enc_r2 = train_and_evaluate(
    pipeline_enc, X_train, y_train, X_test, y_test, "ElasticNet")

best_model = "Lasso" if lasso_r2 > enc_r2 else "ElasticNet"
print(f"🔷 Best model based on R² score: {best_model}")

Lasso R² score: 0.5045
ElasticNet R² score: 0.4736
🔷 Best model based on R² score: Lasso


# 6. MLflow logging

In [14]:
# Get data version for tracking
data_version = get_data_version(df_silver)

# Get metrics of trained models
lasso_metrics = create_comprehensive_metrics(
    lasso_model, X_train, y_train, y_test, lasso_pred
)

enc_metrics = create_comprehensive_metrics(
    enc_model, X_train, y_train, y_test, enc_pred
)

# Log trained models
log_model(lasso_model, "Lasso", lasso_metrics, X_test, data_version)
log_model(enc_model, "ElasticNet", enc_metrics, X_test, data_version)

# Compare and log best model
models_dict = {
    "Lasso": {"pipeline": lasso_model, "metrics": lasso_metrics},
    "ElasticNet": {"pipeline": enc_model, "metrics": enc_metrics}
}

best_model = compare_and_log_models(models_dict, data_version)

Registered model 'sk-learn-Lasso-model' already exists. Creating a new version of this model...
Created version '2' of model 'sk-learn-Lasso-model'.


✅ Lasso logged (R² = 0.5045)
✅ ElasticNet logged (R² = 0.4736)
🔷 Best model: Lasso (R² = 0.5045)


Registered model 'sk-learn-ElasticNet-model' already exists. Creating a new version of this model...
Created version '2' of model 'sk-learn-ElasticNet-model'.
